# 4. Feature Engineering

In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/processed/train_processed.csv")
val_df = pd.read_csv("../data/processed/validation_processed.csv")
test_df = pd.read_csv("../data/processed/test_processed.csv")

print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Training shape: (15999, 3)
Validation shape: (2000, 3)
Test shape: (2000, 3)


In [2]:
print("Training columns:", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

Training columns: ['text', 'label', 'clean_text']
Validation columns: ['text', 'label', 'clean_text']
Test columns: ['text', 'label', 'clean_text']


In [3]:
#  Separating Text Features and Labels

X_train_text = train_df["clean_text"]
y_train = train_df["label"]

X_val_text = val_df["clean_text"]
y_val = val_df["label"]

X_test_text = test_df["clean_text"]
y_test = test_df["label"]

print("Training text samples:", len(X_train_text))
print("Training labels:", len(y_train))

print("Validation text samples:", len(X_val_text))
print("Validation labels:", len(y_val))

print("Test text samples:", len(X_test_text))
print("Test labels:", len(y_test))

Training text samples: 15999
Training labels: 15999
Validation text samples: 2000
Validation labels: 2000
Test text samples: 2000
Test labels: 2000


In [4]:
#  Verifying Label Consistency

print("Training labels:", sorted(y_train.unique()))
print("Validation labels:", sorted(y_val.unique()))
print("Test labels:", sorted(y_test.unique()))

Training labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Validation labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Test labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [5]:
# Initializing the TF-IDF Vectorizer

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

print(tfidf_vectorizer)

TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2), sublinear_tf=True)


In [ ]:
# ngram_range=(1, 2)   → unigrams + bigrams
# min_df=2             → removes extremely rare terms
# max_df=0.95          → removes extremely common terms
# sublinear_tf=True    → logarithmic term-frequency scaling

In [6]:
# Fitting TF-IDF on the Training Data

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Number of TF-IDF features:", X_train_tfidf.shape[1])

Training TF-IDF shape: (15999, 32992)
Number of TF-IDF features: 32992


In [7]:
#  Transforming Validation and Test Data

X_val_tfidf = tfidf_vectorizer.transform(X_val_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Validation TF-IDF shape: (2000, 32992)
Test TF-IDF shape: (2000, 32992)


In [8]:
#  Inspecting the TF-IDF Vocabulary

feature_names = tfidf_vectorizer.get_feature_names_out()

print("Total features:", len(feature_names))
print("\nFirst 30 features:")
print(feature_names[:30])

Total features: 32992

First 30 features:
['aa' 'aa meeting' 'abandon' 'abandoned' 'abandoning' 'abandonment' 'abc'
 'abdomen' 'abide' 'abilities' 'abilities and' 'abilities as' 'ability'
 'ability to' 'abit' 'able' 'able to' 'about' 'about all' 'about an'
 'about and' 'about animals' 'about any' 'about anything' 'about as'
 'about beauty' 'about because' 'about being' 'about books' 'about but']


In [9]:
#  Inspecting TF-IDF Feature Values

print("TF-IDF matrix type:", type(X_train_tfidf))
print("Number of non-zero values:", X_train_tfidf.nnz)

print("\nSample TF-IDF values from the first document:")
print(X_train_tfidf[0].toarray()[0][:20])

TF-IDF matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Number of non-zero values: 412026

Sample TF-IDF values from the first document:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [10]:
#  Inspecting Non-Zero TF-IDF Features

first_row = X_train_tfidf[0]

nonzero_indices = first_row.nonzero()[1]
nonzero_values = first_row.data

first_document_features = sorted(
    zip(
        feature_names[nonzero_indices],
        nonzero_values
    ),
    key=lambda x: x[1],
    reverse=True
)

print("Original text:")
print(X_train_text.iloc[0])

print("\nNon-zero TF-IDF features:")
for feature, value in first_document_features:
    print(f"{feature}: {value:.4f}")

Original text:
i didnt feel humiliated

Non-zero TF-IDF features:
feel humiliated: 0.5801
humiliated: 0.5262
didnt feel: 0.4649
didnt: 0.3980
feel: 0.1096


In [11]:
#  Measuring TF-IDF Sparsity

total_elements = X_train_tfidf.shape[0] * X_train_tfidf.shape[1]
nonzero_elements = X_train_tfidf.nnz
zero_elements = total_elements - nonzero_elements

sparsity_percentage = (zero_elements / total_elements) * 100

print("Total matrix elements:", total_elements)
print("Non-zero elements:", nonzero_elements)
print("Zero elements:", zero_elements)
print(f"Sparsity: {sparsity_percentage:.2f}%")
print(f"Density: {(100 - sparsity_percentage):.2f}%")

Total matrix elements: 527839008
Non-zero elements: 412026
Zero elements: 527426982
Sparsity: 99.92%
Density: 0.08%


#### Saving the TF-IDF Vectorizer

The fitted TF-IDF vectorizer is saved so that the exact vocabulary and IDF weights learned from the training data can be reused during model training and future predictions.

Saving the fitted vectorizer also ensures that new text is transformed using the same feature representation as the training data.

In [12]:
#  Saving the TF-IDF Vectorizer

import os
import joblib

os.makedirs("../models", exist_ok=True)

vectorizer_path = "../models/tfidf_vectorizer.pkl"

joblib.dump(
    tfidf_vectorizer,
    vectorizer_path
)

print("TF-IDF vectorizer saved:", vectorizer_path)
print("File exists:", os.path.exists(vectorizer_path))

TF-IDF vectorizer saved: ../models/tfidf_vectorizer.pkl
File exists: True


In [13]:
#  Saving TF-IDF Feature Matrices


from scipy.sparse import save_npz

os.makedirs("../data/processed", exist_ok=True)

save_npz(
    "../data/processed/X_train_tfidf.npz",
    X_train_tfidf
)

save_npz(
    "../data/processed/X_val_tfidf.npz",
    X_val_tfidf
)

save_npz(
    "../data/processed/X_test_tfidf.npz",
    X_test_tfidf
)

np.save(
    "../data/processed/y_train.npy",
    y_train.to_numpy()
)

np.save(
    "../data/processed/y_val.npy",
    y_val.to_numpy()
)

np.save(
    "../data/processed/y_test.npy",
    y_test.to_numpy()
)

print("TF-IDF feature matrices and labels saved successfully.")

TF-IDF feature matrices and labels saved successfully.


In [14]:
#  Verifying Saved Feature Artifacts

saved_files = [
    "../models/tfidf_vectorizer.pkl",
    "../data/processed/X_train_tfidf.npz",
    "../data/processed/X_val_tfidf.npz",
    "../data/processed/X_test_tfidf.npz",
    "../data/processed/y_train.npy",
    "../data/processed/y_val.npy",
    "../data/processed/y_test.npy"
]

for file_path in saved_files:
    print(
        os.path.basename(file_path),
        "->",
        "Exists" if os.path.exists(file_path) else "NOT FOUND"
    )

tfidf_vectorizer.pkl -> Exists
X_train_tfidf.npz -> Exists
X_val_tfidf.npz -> Exists
X_test_tfidf.npz -> Exists
y_train.npy -> Exists
y_val.npy -> Exists
y_test.npy -> Exists


In [15]:
# Final Feature Validation

from scipy.sparse import load_npz

X_train_check = load_npz("../data/processed/X_train_tfidf.npz")
X_val_check = load_npz("../data/processed/X_val_tfidf.npz")
X_test_check = load_npz("../data/processed/X_test_tfidf.npz")

y_train_check = np.load("../data/processed/y_train.npy")
y_val_check = np.load("../data/processed/y_val.npy")
y_test_check = np.load("../data/processed/y_test.npy")

print("Training features:", X_train_check.shape)
print("Validation features:", X_val_check.shape)
print("Test features:", X_test_check.shape)

print("\nTraining labels:", y_train_check.shape)
print("Validation labels:", y_val_check.shape)
print("Test labels:", y_test_check.shape)

print("\nFeature type:", type(X_train_check))

Training features: (15999, 32992)
Validation features: (2000, 32992)
Test features: (2000, 32992)

Training labels: (15999,)
Validation labels: (2000,)
Test labels: (2000,)

Feature type: <class 'scipy.sparse._csr.csr_matrix'>


##  Feature Engineering Summary

The feature engineering stage has been completed successfully.

### Key steps

1. Loaded the processed training, validation, and test datasets.
2. Separated the normalized text from the target labels.
3. Verified that all six emotion classes are consistently represented across all dataset splits.
4. Initialized a TF-IDF vectorizer using word unigrams and bigrams.
5. Fitted the TF-IDF vectorizer exclusively on the training data to prevent data leakage.
6. Transformed the validation and test datasets using the fitted training vectorizer.
7. Generated 32,992 TF-IDF features.
8. Verified that the TF-IDF representation is highly sparse, with 99.92% sparsity.
9. Saved the fitted TF-IDF vectorizer.
10. Saved the sparse TF-IDF feature matrices and corresponding labels.
11. Reloaded the saved artifacts and verified their shapes and data types.

### Final Feature Dimensions

- Training: 15,999 × 32,992
- Validation: 2,000 × 32,992
- Test: 2,000 × 32,992

### Saved Artifacts

- `models/tfidf_vectorizer.pkl`
- `data/processed/X_train_tfidf.npz`
- `data/processed/X_val_tfidf.npz`
- `data/processed/X_test_tfidf.npz`
- `data/processed/y_train.npy`
- `data/processed/y_val.npy`
- `data/processed/y_test.npy`

The TF-IDF features are now ready for supervised machine learning model development.